# Cleaning downloaded Data from GISAID

Author: Alexander Maksiaev

Purpose: Download and clean GISAID data, after de-duplicating from Andersen/NCBI Virus data

Notes: 
* The "downloads" folder MUST be your computer's downloads folder, or wherever your browser automatically downloads files. This folder must also be cleaned in between each run of this code.
* The returned files from this code will be stored in a separate folder after running -- no other action is needed, aside from cleaning the original downloads folder after this code runs.  
* This file MUST be in the same folder as "utils.py"

## Housekeeping ##

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import dateutil
import openpyxl
from itertools import islice
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [2]:
# Paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
andersen_ncbi_virus_gisaid = home + "Combinations/NCBI_Virus_Andersen_GISAID/" 

os.chdir(downloads)

## Collect user input

In [3]:
# locations = input("Locations (separate with commas and no spaces): ")
# start_date = input("Start date (format: YYYY-MM-DD): ")
# end_date = input("End date (format: YYYY-MM-DD): ")

In [4]:
locations = "Antarctica,North America,South America"
# locations = "Antarctica,South America"
start_date = "2021-11-01"
end_date = "2025-12-05"

## Create directories if needed

In [5]:
downloads_saved = home + "GISAID/downloads/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
andersen_ncbi_virus = home + "Combinations/NCBI_Virus_Andersen/" + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
gisaid_files = home + "GISAID/complete/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
complete_files = andersen_ncbi_virus_gisaid + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"

if not os.path.exists(gisaid_files): # checking if the directory exists or not
    os.makedirs(gisaid_files) # if the directory is not present then create it

if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

print(complete_files)

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/Combinations/NCBI_Virus_Andersen_GISAID/11-01-2021--12-05-2025_Antarctica_North_America_South_America/


## Get list of genotypes and states

In [6]:
# Get list of genotypes and states

# os.chdir("C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/")
os.chdir(references)

states = pd.read_csv("states_ref.csv")

genotypes_df = pd.read_excel("genotype_key.xlsx")

genotypes = list(genotypes_df["Genotype"])
genotypes.append("Unassigned")

# print(genotypes)

# genotypes = ["B3.2", "B3.6", "B3.7", "B3.5", "A3", "B3.13", "D1.1", "D1.3"]

# genotypes = ["B3.2"]

# genotypes = ["B3.13", "D1.1", "D1.3"] # , "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

## Download all files, run through all files, convert fasta files to dataframes, and separate them into different dataframes based on segment ##

In [ ]:
all_metadata_files = []
all_fasta_files = []

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it
    # Move downloaded files to saved downloads
    for dirpath, dirs, files in os.walk(downloads):
        if len(files) > 0: # If we have any files that need to be moved
            for file in files:
                file_name = os.path.join(dirpath, file)
                destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                try:
                    shutil.move(file_name, destination_path)
                except:
                    print("Error moving file", file_name)
                    continue 
        else: # If we don't have any downloaded files
            # Have user type in username and password
            username = input("Username: ")
            password = input("Password: ")
            browser = input("Browser: ")
            sleep_time = input("Seconds to sleep in between clicks: ")

            open_gisaid(username, password, browser, sleep_time, locations, start_date, end_date) # Download files -- NOT WORKING RIGHT NOW

            # Re-try 
            for dirpath, dirs, files in os.walk(downloads):
                if len(files) > 0: # If we have any files that need to be moved
                    for file in files:
                        file_name = os.path.join(dirpath, file)
                        destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                        try:
                            shutil.move(file_name, destination_path)
                        except:
                            print("Error moving file", file_name)
                            continue 
                break 
        break 


for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # file_name = "_".join(file_name.split(" "))
        
        os.rename(file_name, "_".join(file_name.split(" ")).replace("(", "").replace(")", ""))

# PAUSE AND ARCHIVE XLS FILE AFTER SAVING AS XLSX
for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)
        print(file_name)
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name)
            all_metadata_files.append(metadata)
        # if ".csv" in file_name:
        #     metadata = pd.read_csv(file_name)
        #     all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states) # Convert fasta file to dataframe
            # print(fasta_file[fasta_file["Geo_Location"] != "USA"])
            # break 
            all_fasta_files.append(fasta_file)
    break 

print(len(all_metadata_files))
print(len(all_fasta_files))

# print(all_metadata_files)

# all_metadata_files = [all_metadata_files[0]]
# all_fasta_files = [all_fasta_files[1]]

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/2021-11-01--2025-12-05_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/2021-11-01--2025-12-05_Antarctica_North_America_South_America/gisaid_epiflu_sequence.fasta


In [ ]:
# Get metadata

def separate_fasta_by_segs(metadata, fasta, animals_df, genotypes): #, b313_fasta, d11_fasta):

    fasta = fix_animals(fasta, animals_df) # Fix animals first
    # Dummy host type -- we'll actually add this in later
    # b313_fasta["Host_Type"] = "other"
    # d11_fasta["Host_Type"] = "other"

    unique_segments = list(set(fasta["Segment"])) # Get list of segments
    # genotypes = ["B3.13", "D1.1"]
    # genotype_fastas = {"B3.13": b313_fasta, "D1.1": d11_fasta}

    # “>EPI_ID/Isolate_name|subtype|collection_date|host_type|genotype”

    segment_fastas = [] # Get a list of fastas, separated by segment
    for fasta_gen in genotypes: # .keys(): # For each genotype
        print(fasta_gen)
        for seg in unique_segments: # For each segment

            xls = metadata[metadata["Genotype"].apply(lambda x: x.split(" ")[0]) == fasta_gen] # Get only the metadata corresponding to that genotype

            # print(xls)
            # print("XLS: ", metadata["Genotype"])
            # print(xls["Isolate_Id"])

            # print(d11_xls)

            # FASTA
            # if "Identifier" in fasta.columns:
            #     mask = fasta["Identifier"].isin(xls['Isolate_Id'])
            # else:           
            #     mask = fasta['Isolate_Id'].isin(xls['Isolate_Id'])

            fasta_seg_pre = fasta[fasta["Identifier"].isin(xls['Isolate_Id'])] # Get only the identifiers (Isolate_Id) that are left after metadata is filtered for genotype
            # print(fasta_seg_pre)

            # fasta_seg = genotype_fastas[fasta_gen][genotype_fastas[fasta_gen]["Segment"] == seg]
            fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

            fasta_seg["Genotype"] = fasta_gen

            # Rename sequences 
            new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name"] + "|" + fasta_seg["Subtype"] + "|" + fasta_seg["Geo_Location"] + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_seg["Host_Type"] + "|" + fasta_seg["Genotype"] + "\n"
            fasta_seg["New_Name"] = new_name
            # print(fasta_seg["New_Name"])

            segment_fastas.append(fasta_seg)
            # print(fasta_seg)
    for seg in unique_segments:
        xls = metadata[metadata["Genotype"].str.contains("Not assigned")] # Get only the metadata corresponding to that genotype

        # print(xls)
        # print("XLS: ", metadata["Genotype"])
        # print(xls["Isolate_Id"])

        # print(d11_xls)

        # FASTA
        # if "Identifier" in fasta.columns:
        #     mask = fasta["Identifier"].isin(xls['Isolate_Id'])
        # else:           
        #     mask = fasta['Isolate_Id'].isin(xls['Isolate_Id'])

        fasta_seg_pre = fasta[fasta["Identifier"].isin(xls['Isolate_Id'])] # Get only the identifiers (Isolate_Id) that are left after metadata is filtered for genotype
        # print(fasta_seg_pre)

        # fasta_seg = genotype_fastas[fasta_gen][genotype_fastas[fasta_gen]["Segment"] == seg]
        fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

        fasta_seg["Genotype"] = "Unassigned"

        # Rename sequences 
        new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name"] + "|" + fasta_seg["Subtype"] + "|" + fasta_seg["Geo_Location"] + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_seg["Host_Type"] + "|" + fasta_seg["Genotype"] + "\n"
        fasta_seg["New_Name"] = new_name
        # print(fasta_seg["New_Name"])

        segment_fastas.append(fasta_seg)

    return segment_fastas, unique_segments


# Separate fastas by segment
segment_fastas = []
unique_animals_all = []
for i, fasta in enumerate(all_fasta_files):
    # print(all_fasta_files)
    # print(fasta)
    # print(i)
    metadata = all_metadata_files[i]
    # print(metadata)
    # print(fasta.loc[i, "Isolate_Name"])
    
    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    os.chdir(references)
    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_segs(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment
    # print(fastas[0])

    for fasta in fastas:
        segment_fastas.append(fasta)

# print(segment_fastas[0][0][segment_fastas[0][0]["Genotype"] == "D1.1"])

A1


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

A2


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

A3


...

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

Minor102
Minor103
Minor104


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

Minor105
Minor106
Unassigned


C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["Genotype"] = fasta_gen
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:45: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasta_seg["New_Name"] = new_name
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_28932\2834623338.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .l

## De-Duplication

In [ ]:
# Get files from Andersen and NCBI Virus

# Grab files
andersen_ncbi = {}
for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
        fasta_file = fasta_df_complete(file_name, states) # Convert fasta file to dataframe
        andersen_ncbi[segment_genotype] = fasta_file
    
# print(andersen_ncbi["B3.13_HA"])

In [ ]:
# Do all segments, not just HA (test)

# Check isolate IDs to see if they already exist in Andersen/NCBI
# Match based on year AND partial isolate ID, as some partials may be identical between years

gisaid_list = [] 
for gisaid_fasta in segment_fastas:
    # for genotype_gisaid_fasta in gisaid_fasta:
    # print(genotype_gisaid_fasta)
    gisaid_fasta["Partials"] = gisaid_fasta["Isolate_Id"].apply(partial_isolate)
    gisaid_fasta["Year"] = gisaid_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x).year))
    # genotype_gisaid_fasta = genotype_gisaid_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
    gisaid_list.append(gisaid_fasta)
    # print(genotype_gisaid_fasta)
        
# print(ha_only)
    
andersen_ncbi_genotypes = {}
for key in andersen_ncbi:
    andersen_ncbi_fasta = andersen_ncbi[key]
    print(andersen_ncbi_fasta)
    # Find partial Isolate IDs
    andersen_ncbi_fasta["Partials"] = andersen_ncbi_fasta["Isolate_Id"].apply(partial_isolate)
    # print("Andersen:", andersen_ncbi_fasta["Partials"])
    andersen_ncbi_fasta["Year"] = andersen_ncbi_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if len(x) > 0 else x)
    andersen_ncbi_fasta["Segment"] = key.split("_")[-1]
    # andersen_ncbi_fasta = andersen_ncbi_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
    # print(andersen_ncbi_fasta)
    if andersen_ncbi_fasta["Genotype"].values[0] not in andersen_ncbi_genotypes.keys(): # If we haven't already seen this genotype
        andersen_ncbi_genotypes[andersen_ncbi_fasta["Genotype"].values[0]] = andersen_ncbi_fasta # Add fasta to genotype dictionary
        # print(andersen_ncbi_fasta)
    # break 

# print(andersen_ncbi_genotypes)

# gisaid_only_dfs = {} # We need these so that we know what sequences to throw out in the other segments too
throw_away = {}
# gisaid_dups = []
# andersen_ncbi_dups = []
counter = 0
for gisaid_genotype in gisaid_list: # Each dataframe is unique in genotype
    # print(gisaid_genotype[gisaid_genotype["Host_Type"] == "human"])
    if len(gisaid_genotype["Genotype"]) > 0:
        genotype = gisaid_genotype["Genotype"].values[0]
        andersen_ncbi_fasta = pd.DataFrame()
        if genotype in andersen_ncbi_genotypes.keys(): # and genotype not in gisaid_only_dfs.keys(): # If it's in Andersen and we haven't seen it before here
            andersen_ncbi_fasta = andersen_ncbi_genotypes[genotype] # Get the dataframe with the same genotype
            # print(len(andersen_ncbi_fasta))
            # print(len(gisaid_genotype))
            # deduplicated = pd.concat([gisaid_genotype,andersen_ncbi_fasta]).drop_duplicates(subset=["Partials", "Year"], keep="last") 
            gisaid_duplicates = gisaid_genotype[gisaid_genotype.duplicated(["Partials", "Year"], keep=False)]
            # print(gisaid_duplicates)
            andersen_ncbi_fasta_duplicates = andersen_ncbi_fasta[andersen_ncbi_fasta.duplicated(["Partials", "Year"], keep=False)]
            to_throw = pd.concat([gisaid_genotype, andersen_ncbi_fasta])[pd.concat([gisaid_genotype, andersen_ncbi_fasta]).duplicated(["Partials", "Year"], keep=False)]
            # print(len(deduplicated))
            # print(len(to_throw))
            # print(to_throw)
            os.chdir(andersen_ncbi_virus_gisaid)
            to_throw.to_csv(str(counter) + "_throw_away.csv")
            # throw_away[genotype] = to_throw
            between_duplicates = []
            for t in to_throw["Identifier"].values:
                if t not in gisaid_duplicates and t not in andersen_ncbi_fasta_duplicates:
                    # print(t)
                    between_duplicates.append(t)
            if genotype in throw_away.keys(): # If there's already a genotype
                throw_away[genotype] += between_duplicates
            else:
                throw_away[genotype] = between_duplicates
        else:
            print("GISAID genotype:", gisaid_genotype)
            throw_away[genotype] = []
    counter += 1
            # gisaid_dups.append(gisaid_duplicates)
            # andersen_ncbi_dups.append(andersen_ncbi_fasta)
            # gisaid_only_dfs[genotype] = deduplicated
        # else:
        #     gisaid_only_dfs[genotype] = gisaid_genotype
    # else:
    #     deduplicated = gisaid_genotype
    
# print(gisaid_only_dfs["D1.3"][gisaid_only_dfs["D1.3"]["Host_Type"] == "cattle"])
    
        # print(deduplicated[deduplicated["Host_Type"] == "human"])

# print(gisaid_only_dfs)

# Identify sequences we are keeping
# genotype_seq_keep = {}
# for gisaid_genotype in gisaid_list:

#     to_throw = throw_away[genotype]
#     genotype_seq_keep[genotype] = list(deduplicated["Identifier"])
for key in throw_away:
    print(key)
    print(len(throw_away[key]))
# Keep in other segments only the sequences we kept in HA
kept_seqs = []
print(len(segment_fastas))
for gisaid_df in segment_fastas:
    # print(len(genotype_group))
    # for gisaid_df in genotype_group:
        # print(gisaid_df)
    if len(gisaid_df["Genotype"].dropna()) > 0: # If there are sequences
        genotype = gisaid_df["Genotype"].values[0]
        # print(len(genotype_seq_keep[genotype]))
        # gisaid_df_new = gisaid_df[gisaid_df['Identifier'].isin(genotype_seq_keep[genotype])]
        if genotype in throw_away:
            to_throw = throw_away[genotype]
            print("Original length:", genotype, len(gisaid_df))
            gisaid_df_new = gisaid_df[~gisaid_df['Identifier'].isin(to_throw)]
            print("New length:", len(gisaid_df_new))
            kept_seqs.append(gisaid_df_new)
        # else:


# print(len(kept_seqs))

                                                Header       Isolate_Id  \
0    |A/American_Crow/NS/FAV-0322-1-2022/2022|H5N1|...  FAV-0322-1-2022   
1    |A/American_Crow/NS/FAV-0467-1-2022/2022|H5N1|...  FAV-0467-1-2022   
2    |A/American_Crow/PE/FAV-0109-1-2022/2022|H5N1|...  FAV-0109-1-2022   
3    |A/American_Crow/QC/FAV-0855-4-2022/2022|H5N1|...  FAV-0855-4-2022   
4    |A/American_Crow/QC/FAV-0855-5-2022/2022|H5N1|...  FAV-0855-5-2022   
..                                                 ...              ...   
508  |A/chicken/Wisconsin/22-007545-001/2022|H5N1|U...    22-007545-001   
509  |A/chicken/Wisconsin/22-007545-004/2022|H5N1|U...    22-007545-004   
510  |A/chicken/Wisconsin/22-007545-005/2022|H5N1|U...    22-007545-005   
511  |A/goose/Missouri/22-012338-006/2022|H5N1|USA-...    22-012338-006   
512  |A/American_wigeon/South_Carolina/AH0195145/20...        AH0195145   

                                        Isolate_Name Subtype         Partials  \
0            A/Ame

In [ ]:
# # Check isolate IDs to see if they already exist in Andersen/NCBI
# # Match based on year AND partial isolate ID, as some partials may be identical between years

# ha_only = [] # Only do one segment, as the others are identical 
# for gisaid_fasta in segment_fastas:
#     for genotype_gisaid_fasta in gisaid_fasta:
#         # print(genotype_gisaid_fasta)
#         if "HA" in genotype_gisaid_fasta["Segment"].values:
#             genotype_gisaid_fasta["Partials"] = genotype_gisaid_fasta["Isolate_Id"].apply(partial_isolate)
#             genotype_gisaid_fasta["Year"] = genotype_gisaid_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x).year))
#             genotype_gisaid_fasta = genotype_gisaid_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
#             ha_only.append(genotype_gisaid_fasta)
#         # print(genotype_gisaid_fasta)
        
# # print(ha_only)
    
# andersen_ncbi_genotypes = {}
# for key in andersen_ncbi:
#     andersen_ncbi_fasta = andersen_ncbi[key]
#     # Find partial Isolate IDs
#     andersen_ncbi_fasta["Partials"] = andersen_ncbi_fasta["Isolate_Id"].apply(partial_isolate)
#     # print("Andersen:", andersen_ncbi_fasta["Partials"])
#     andersen_ncbi_fasta["Year"] = andersen_ncbi_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year))
#     andersen_ncbi_fasta["Segment"] = key.split("_")[-1]
#     andersen_ncbi_fasta = andersen_ncbi_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
#     # print(andersen_ncbi_fasta)
#     if andersen_ncbi_fasta["Genotype"].values[0] not in andersen_ncbi_genotypes.keys() and andersen_ncbi_fasta["Segment"].values[0] == "HA": # If we haven't already seen this genotype, and if HA
#         andersen_ncbi_genotypes[andersen_ncbi_fasta["Genotype"].values[0]] = andersen_ncbi_fasta # Add fasta to genotype dictionary
#         # print(andersen_ncbi_fasta)
#     # break 

# # print(andersen_ncbi_genotypes)

# gisaid_only_dfs = {} # We need these so that we know what sequences to throw out in the other segments too
# for gisaid_genotype in ha_only: # Each dataframe is unique in genotype
#     # print(gisaid_genotype[gisaid_genotype["Host_Type"] == "human"])
#     genotype = gisaid_genotype["Genotype"].values[0]
#     andersen_ncbi_fasta = pd.DataFrame()
#     if genotype in andersen_ncbi_genotypes.keys(): # and genotype not in gisaid_only_dfs.keys(): # If it's in Andersen and we haven't seen it before here
#         andersen_ncbi_fasta = andersen_ncbi_genotypes[genotype] # Get the dataframe with the same genotype
#         print(len(andersen_ncbi_fasta))
#         print(len(gisaid_genotype))
#         deduplicated = pd.concat([gisaid_genotype,andersen_ncbi_fasta]).drop_duplicates(subset=["Partials", "Year"], keep="last") 
#         print(len(deduplicated))
#         gisaid_only_dfs[genotype] = deduplicated
#     # else:
#     #     deduplicated = gisaid_genotype
    
# # print(gisaid_only_dfs["D1.3"][gisaid_only_dfs["D1.3"]["Host_Type"] == "cattle"])
    
#         # print(deduplicated[deduplicated["Host_Type"] == "human"])

# # print(gisaid_only_dfs)

# # Identify sequences we are keeping
# genotype_seq_keep = {}
# for genotype in gisaid_only_dfs:
#     deduplicated = gisaid_only_dfs[genotype]
#     genotype_seq_keep[genotype] = list(deduplicated["Identifier"])

# # Keep in other segments only the sequences we kept in HA
# kept_seqs = []
# for genotype_group in segment_fastas:
#     # print(genotype_group)
#     for gisaid_df in genotype_group:
#         # print(gisaid_df)
#         if len(gisaid_df["Genotype"]) > 0: # If there are sequences
#             genotype = gisaid_df["Genotype"].values[0]
#             print(len(genotype_seq_keep[genotype]))
#             gisaid_df_new = gisaid_df[gisaid_df['Identifier'].isin(genotype_seq_keep[genotype])]
#             print(len(gisaid_df_new))
#             kept_seqs.append(gisaid_df_new)

# print(len(kept_seqs))

Create animal reference if needed 

In [ ]:
# Find animals to sort, if needed

os.chdir(references)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(home)
animals_df.to_csv("animals_ref_to_sort.csv")

['numida_meleagris', 'bufflehead_duck', 'louisiana', 'lesser_white-fronted_goose', 'snowy_owl', 'american_crow', 'atlantic_white-sided_dolphin', "franklin's_gull", 'dovekie', 'cinnamon_teal', 'laughing_gull', 'manx_shearwater', 'wild_goose', 'king_penguin', 'backyard_turkey', 'crested_jay', 'house_sparrow', 'american_raven', 'lesser_scaup', 'duck', 'short_eared_owl', 'ring-billed_gull', 'guinea_fowl', 'pinniped', 'peafowl', 'dog', "swanson's_hawk", 'ringed_seal', 'franklin_gull', 'black-billed_magpie', 'glaucous-winged_gull', 'sparrow', 'mottled_duck', 'savannah_cat', 'colorado', 'sharp-shinned_hawk', 'swallow', 'unknown', 'whooping_crane', 'rhea', 'turkey_vulture', 'chilean_teal', 'peregrine_falcon', 'eagle', 'skunk', 'wild_mink', 'gray_seal', 'broiler_chicken', 'ruddy_duck', 'short-billed_gull', 'red-shouldered_hawk', 'swan', 'great-tailed_grackle', 'cormorant', 'wyoming', 'tern', 'rough-legged_hawk', 'south_polar_skua', 'american_wood_stork', 'canada_goose', 'greater_yellowlegs', 's

## Merge all fastas into different files -- 8 segments * X genotypes ##

In [ ]:
# 16 files needed
huge_fasta = pd.DataFrame()

for fastas in kept_seqs: # 7 batches
    # print(fastas.columns)
    # print(len(fastas))
    # break
    # for f in fastas: # 16 files per batch 
        # print(f)
        # break 
    huge_fasta = pd.concat([huge_fasta, fastas])

print(huge_fasta.columns)

# Now separate huge_fasta into 16 fastas
big_fastas = []

# print(huge_fasta)

genotypes.append("Unassigned")
for gen in genotypes:
    # print(gen)
    big_fasta = huge_fasta[huge_fasta["Genotype"] == gen]
    # print(gen)
    for seg in unique_segments:
        seg_specific_fasta = big_fasta[big_fasta["Segment"] == seg]
        big_fastas.append(seg_specific_fasta)

print(big_fasta)

# Now that we have 16 fastas, write the files
for fasta in big_fastas:

    # Create a dictionary to create a file
    fasta_df = fasta[["New_Name", "Sequence"]]
    fasta_dict = pd.Series(fasta_df.Sequence.values,index=fasta_df.New_Name).to_dict()
    # print(fasta["Genotype"])
    # Create fasta file 
    try: 
        output_path = gisaid_files + fasta["Genotype"].values[0] + "_" + fasta["Segment"].values[0] + "_GISAID_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            item = item.replace(" ", "_")
            # Fix broken sequences
            # if "EPI_ISL_20151597" in item:
            #     item = "EPI_ISL_20151597|A/Brown_Skua/Gough_Island/047355/2024|H5N1|Antarctica|2024-09-20|wild_avian|B3.2"
            # elif "EPI_ISL_20151596" in item:
            #     item = "EPI_ISL_20151596|A/Brown_Skua/Gough_Island/047354/2024|H5N1|Antarctica|2024-09-20|wild_avian|B3.2"
            output_file.write(item)
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype"].values[0])
        output_file.close()
    except:
        # print(fasta["Genotype"])
        # print(fasta)
        print("Could not find any results for genotype.")
        # continue

print(len(huge_fasta))
print(len(big_fastas[0]))

Index(['Header', 'Isolate_Id', 'Isolate_Name', 'Subtype', 'Segment',
       'Location', 'Geo_Location', 'Date Collected', 'Species', 'Sequence',
       'Identifier', 'Host_Type', 'Genotype', 'New_Name', 'Partials', 'Year'],
      dtype='object')
                                                   Header  \
2189    EPI_ISL_19660878|A/mallard/Wisconsin/24-023158...   
2197    EPI_ISL_19660872|A/mallard/Wisconsin/24-023158...   
2213    EPI_ISL_19660875|A/mallard/Wisconsin/24-023158...   
2229    EPI_ISL_19660869|A/mallard/Wisconsin/24-023158...   
2373    EPI_ISL_19660989|A/mallard/Ohio/24-022862-032-...   
...                                                   ...   
155820  EPI_ISL_19429958|A/mallard/North_Dakota/23-033...   
155972  EPI_ISL_19429971|A/mallard/Indiana/23-034359-0...   
156012  EPI_ISL_19429975|A/mallard/Michigan/23-034360-...   
156852  EPI_ISL_19167938|A/environment/Chile/C63181/20...   
157141  EPI_ISL_14777716|A/Sanderling/Delaware/518/202...   

                    I

## Concatenate to Andersen_NCBI files and save

In [ ]:
# Concat
os.chdir(andersen_ncbi_virus_gisaid)

common_genotypes = set()
for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # print(file_name)
        for dirpath1, dirs1, files1 in os.walk(gisaid_files):
            for file1 in files1:
                file_name1 = os.path.join(dirpath1, file1)
                # print(file_name1)
                
                if "_".join(file_name.split("/")[-1].split("_")[0:2]) == "_".join(file_name1.split("/")[-1].split("_")[0:2]): # If they match
                    common_genotypes.add("_".join(file_name.split("/")[-1].split("_")[0]))
                    print("_".join(file_name.split("/")[-1].split("_")[0:2]))
                    output_path = complete_files + "all_" + "_".join(file_name.split("/")[-1].split("_")[0:2]) + "_" + start_date + "--" + end_date + ".fasta" # file_name.split("/")[-1] # Genotype and Segment should all be the same
                    output_file = open(output_path, "w")
                    with open(file_name) as f:
                        for line in f.readlines():
                            output_file.write(line)
                            # output_file.write("\n")
                        f.close()
                    with open(file_name1) as f1:
                       for line in f1.readlines():
                            output_file.write(line)
                            # output_file.write("\n")
                    f1.close()  

                    output_file.close()
                
                

            break 
    break 

# Include uncommon genotypes
for dirpath1, dirs1, files1 in os.walk(gisaid_files):
    for file1 in files1:
        file_name1 = os.path.join(dirpath1, file1)
        if "_".join(file_name1.split("/")[-1].split("_")[0]) not in common_genotypes:
            output_path = complete_files + "all_" + "_".join(file_name1.split("/")[-1].split("_")[0:2]) + "_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
            output_file = open(output_path, "w")
            with open(file_name1) as f:
                for line in f.readlines():
                    output_file.write(line)
                    # output_file.write("\n")
                f.close()

            output_file.close()
            # common_genotypes.add("_".join(file_name1.split("/")[-1].split("_")[0]))

for dirpath1, dirs1, files1 in os.walk(andersen_ncbi_virus):
    for file1 in files1:
        file_name1 = os.path.join(dirpath1, file1)
        if "_".join(file_name1.split("/")[-1].split("_")[0]) not in common_genotypes:
            output_path = complete_files + "all_" + "_".join(file_name1.split("/")[-1].split("_")[0:2]) + "_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
            output_file = open(output_path, "w")
            with open(file_name1) as f:
                for line in f.readlines():
                    output_file.write(line)
                    # output_file.write("\n")
                f.close()

            output_file.close()

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA
B3.2_MP
B3.2_NA
B3.2_NP
B3.2_NS
B3.2_PA
B3.2_PB1
B3.2_PB2
B3.3_HA
B3.3_MP


In [ ]:
# # Create new labels FOR D1.3 ONLY

# # Create new labels
# os.chdir(home)
# county_info = pd.read_csv("Positive Case Info = Summary Counties.csv")

# print(county_info["County"])
# print(county_info["Event ID"])

# label_mapping = {}
# for id in county_info["Event ID"].values:
#     if id == id: # If not nan
#         label_mapping[id] = county_info[county_info["Event ID"] == id]["County"].iloc[0] + "|" + county_info[county_info["Event ID"] == id]["Site Owner"].iloc[0].replace(" ", "_")

# for dirpath, dirs, files in os.walk(complete_files):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if "D1.3" in file_name:
#             output_file = open(file_name, "r+")
#             with open(file_name) as f:
#                 lines = f.readlines()
#                 for line in lines:
#                     for key in label_mapping:
#                         if key == key and key in line: 
#                             line = line.replace("\n", "") + "|" + label_mapping[key] + "\n"
#                     output_file.write(line)
#                 f.close()
#             output_file.close()
#     break 

In [ ]:
# Create new labels FOR D1.3 ONLY

# Create new labels
os.chdir(home)
new_county_info = pd.read_excel("Positive Case Info  Summary (version 1).xlsx")
og_county_info = pd.read_csv("Positive Case Info = Summary Counties.csv")
sra_info = pd.read_excel("OH-IN_poultry_clade_matched_seq_metadata (version 1).xlsx")

sra_info["Event ID"] = sra_info["SRA Event ID"]

# print(sra_info)
# print(og_county_info["County"])
# print(county_info["HPAI Case Description"])
# print(county_info["Event ID"])

med_county_info = new_county_info.merge(sra_info, on=["Event ID"])

county_info = med_county_info.merge(og_county_info, on=["Event ID", "Site Owner", "Collection Date"], how="left")
# print(county_info)
# print(county_info["County"])
# new_county_info["County"] = np.where(new_county_info["County"] == "",  new_county_info["County"])
county_info["County"] = county_info["HPAI Case Description"].apply(lambda x: # og_county_info.loc[county_info["HPAI Case Description"].str.contains('_'.join(x.split(' ')[:-1])), 'County'] if len(og_county_info.loc[county_info["HPAI Case Description"].str.contains('_'.join(x.split(' ')[:-1])), 'County']) > 0 else 
                                                                   '_'.join(x.split(' ')[:-1]) + "_County_" + og_county_info[og_county_info["County"].str.contains("_".join(x.split(' ')[:-1]))]["County"].values[0].split("_")[-1] if x == x else x)
# print(county_info["Collection Date"])

# print(county_info[county_info["County"].notna()]["County"])

label_mapping = {}
for id in county_info["SRA Accession"].values:
    print(id)
    if id == id and county_info[county_info["SRA Accession"] == id]["County"].values[0] == county_info[county_info["SRA Accession"] == id]["County"].values[0]: # If not nan
        label_mapping[id] = county_info[county_info["SRA Accession"] == id]["County"].iloc[0] + "|" + county_info[county_info["SRA Accession"] == id]["Site Owner"].iloc[0].replace(" ", "_")

print(len(label_mapping))
print(label_mapping)

for dirpath, dirs, files in os.walk(complete_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if "D1.3" in file_name:
            output_file = open(file_name, "r+")
            with open(file_name) as f:
                lines = f.readlines()
                for line in lines:
                    for key in label_mapping:
                        # break 
                        if key in line and len(line.split("|")) < 8: 
                            print(line.split("|")[-3])
                            if dateutil.parser.parse(line.split("|")[-3], default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(line.split("|")[-3], default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day: # If no collection date in line
                                date = county_info[county_info["SRA Accession"] == key]["Collection Date"].apply(lambda x: dateutil.parser.parse(str(x))).values[0]
                                # print(date)
                                split_line = line.split("|")
                                split_line[-3] = dateutil.parser.parse(str(date)).strftime("%Y-%m-%d")
                                line = ("|").join(split_line)
                                print(line)
                                print("date changed")
                            if line.split("|")[3] == "USA":
                                split_line = line.split("|")
                                split_line[3] = "USA-" + county_info[county_info["SRA Accession"] == key]["County"].apply(lambda x: x.split("_")[-1]).values[0]
                                line = ("|").join(split_line)
                            line = line.replace("\n", "") + "|" + label_mapping[key] + "\n"
                            print(line)
                            print("changed")
                    output_file.write(line)
                f.close()
            output_file.close()
    break 

SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR31959950
SRR31959951
SRR31959950
SRR31959951
SRR32125543
SRR32125544
SRR32125543
SRR32125544
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226929
SRR32226930
SRR32226929
SRR32226930
SRR32226927
SRR32226928
SRR32226927
SRR32226928
SRR32254627
SRR32254628
SRR32254627
SRR3